In [3]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()
df.columns

Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label',
       'ideal_intent', 'ideal_tone'],
      dtype='object')

PART 3: Email Assistant Logic (Reuse from Milestone 1)

In [4]:
def email_assistant(email_text):
    text = str(email_text).lower()
    
    if "urgent" in text or "deadline" in text or "submit" in text:
        return "notify", "urgent"
    
    elif "thank you" in text or "thanks" in text:
        return "ignore", "polite"
    
    else:
        return "respond", "neutral"


PART 4: Generate Predictions

In [5]:
predictions = []

for idx, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": idx,
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,0,respond,neutral
1,1,respond,neutral
2,2,respond,neutral
3,3,respond,neutral
4,4,respond,neutral


PART 5: Evaluate accuracy

In [6]:
eval_df = df.copy()
eval_df["predicted_intent"] = pred_df["predicted_intent"]
eval_df["predicted_tone"] = pred_df["predicted_tone"]


In [7]:
eval_df["intent_correct"] = (
    eval_df["predicted_intent"] == eval_df["ideal_intent"]
)

eval_df["tone_correct"] = (
    eval_df["predicted_tone"] == eval_df["ideal_tone"]
)


In [8]:
intent_accuracy = eval_df["intent_correct"].mean() * 100
tone_accuracy = eval_df["tone_correct"].mean() * 100

intent_accuracy, tone_accuracy


(38.5, 77.0)

In [9]:
errors = eval_df[eval_df["intent_correct"] == False]

errors[[
    "body",
    "ideal_intent",
    "predicted_intent"
]].head(10)


,body,ideal_intent,predicted_intent
6,Congratulations! You have been selected as a l...,ignore,respond
8,"Dear user, we detected a login from a new devi...",notify_human,respond
14,Please complete the mandatory training module ...,notify_human,notify
15,"Hi, don't miss our sale with discounts up to 7...",ignore,respond
17,"Hi, don't miss our sale with discounts up to 7...",ignore,respond
18,Please complete the mandatory training module ...,notify_human,notify
19,Please complete the mandatory training module ...,notify_human,notify
20,Notice: Your account will be locked unless ver...,notify_human,respond
21,Please complete the mandatory training module ...,notify_human,notify
23,Security alert: multiple failed login attempts...,notify_human,respond


PART 6: Save Evaluation Output

In [10]:
eval_df.to_csv(
    "../data/milestone2_output_AbhayMulani.csv",
    index=False
)
